In [11]:
import pandas as pd

Load the cleaned csv

In [12]:
df = pd.read_csv("/Users/ishik/Desktop/brooklyn99-sql-analytics/data/All_B99_cleaned.csv",parse_dates=['Airdate'])

In [13]:
import sqlite3

In [14]:
conn = sqlite3.connect(':memory:')

In [15]:
df.to_sql('b99', conn, index=False, if_exists='replace')

153

In [16]:
query_enriched = """
SELECT 
    Season, Episode, Title, Airdate, Rating, "Total Votes", "Episode Description",
    CASE WHEN Rating >= 8.4 THEN 'Masterpiece'
         WHEN Rating >= 7.2 THEN 'Great'
         WHEN Rating >= 6.4 THEN 'Average'
         ELSE 'Weak' END AS episode_category,
    CASE WHEN LOWER(Title) LIKE '%halloween%' OR LOWER("Episode Description") LIKE '%halloween%' THEN 'Holiday'
         WHEN LOWER(Title) LIKE '%christmas%' OR LOWER("Episode Description") LIKE '%christmas%' THEN 'Holiday'
         WHEN LOWER(Title) LIKE '%thanksgiving%' OR LOWER("Episode Description") LIKE '%thanksgiving%' THEN 'Holiday'
         WHEN LOWER(Title) LIKE '%easter%' OR LOWER("Episode Description") LIKE '%easter%' THEN 'Holiday'
         WHEN LOWER(Title) LIKE '%new year%' OR LOWER("Episode Description") LIKE '%new year%' THEN 'Holiday'
         ELSE 'Non-Holiday' END AS is_holiday
FROM b99
"""
df_enriched = pd.read_sql(query_enriched, conn)

In [17]:
running_avg_query = """
SELECT
    Season,
    Episode,
    Title,
    Rating,

    AVG(Rating) OVER(
        PARTITION BY Season
        ORDER BY Episode
        ROWS BETWEEN UNBOUNDED PRECEDING
        AND CURRENT ROW
    ) AS running_avg_rating

FROM b99
"""
running_avg_df = pd.read_sql(running_avg_query,conn)

In [18]:
cult_fav_query = """
WITH ranked AS (SELECT Season,Episode,Title,Rating,"Total Votes",RANK() OVER(ORDER BY "Total Votes" ASC) AS popularity_rank,
       RANK() OVER(ORDER BY Rating ASC) AS appreciation_rank
FROM b99),
popular_vs_appreciation AS (SELECT Season, Episode, Title, Rating, "Total Votes", popularity_rank, appreciation_rank, popularity_rank - appreciation_rank AS rank_difference
FROM ranked)

SELECT Season,Episode,Title,Rating, "Total Votes" , CASE WHEN (popularity_rank - appreciation_rank) <= -50 THEN 'Cult Favorite'
                                                         WHEN (popularity_rank - appreciation_rank) >= 50 THEN 'Popular Voting Choice'
                                                         ELSE 'Balanced' END AS reception_type
FROM popular_vs_appreciation
"""

cult_fav_df = pd.read_sql(cult_fav_query, conn)

In [19]:
season_improvement_query = """
WITH ranked_episodes AS (
    SELECT
        Season,
        Episode,
        Rating,
        ROW_NUMBER() OVER(
            PARTITION BY Season
            ORDER BY Episode
        ) AS first_ep,
        
        ROW_NUMBER() OVER(
            PARTITION BY Season
            ORDER BY Episode DESC
        ) AS last_ep
    FROM b99
),

season_boundary AS (
    SELECT
        Season,
        MAX(CASE WHEN first_ep = 1 THEN Rating END) AS first_rating,
        MAX(CASE WHEN last_ep = 1 THEN Rating END) AS last_rating
    FROM ranked_episodes
    GROUP BY Season
),

improvement AS (
    SELECT
        Season,
        first_rating,
        last_rating,
        last_rating - first_rating AS improvement
    FROM season_boundary
)

SELECT
    Season,
    first_rating,
    last_rating,
    ROUND(improvement,2) AS improvement
FROM improvement
ORDER BY improvement DESC;


"""

season_improvement_df = pd.read_sql(season_improvement_query, conn)

In [20]:
df_enriched.to_csv('/Users/ishik/Desktop/brooklyn99-sql-analytics/exports/b99_episodes_cleaned.csv', index=False)  
running_avg_df.to_csv('/Users/ishik/Desktop/brooklyn99-sql-analytics/exports/b99_running_avg_by_season.csv', index=False)
cult_fav_df.to_csv('/Users/ishik/Desktop/brooklyn99-sql-analytics/exports/b99_cult_favorites.csv', index=False)
season_improvement_df.to_csv('/Users/ishik/Desktop/brooklyn99-sql-analytics/exports/b99_season_improvement.csv', index=False)